# Cambio de monedas

Con monedas ilimitadas de cada tipo, dar una cantidad con el menor número de monedas o contar de cuántas formas se puede dar. La tabla $M[c]$ guarda el mínimo de monedas para la cantidad $c$: $M[0] = 0$ y $M[c] = 1 + \min_{m \le c} M[c - m]$. Cuesta $O(n \cdot C)$ con $n$ tipos de moneda y cantidad $C$.

## 1. Mínimo número de monedas

Con las monedas de euro (en céntimos), calcular el mínimo de monedas para dar una cantidad. Devolver `None` si no se puede.

In [1]:
from math import inf

EUROS = [1, 2, 5, 10, 20, 50, 100, 200]

def tabla_monedas(monedas, cantidad):
    M = [0] + [inf] * cantidad
    for c in range(1, cantidad + 1):
        for m in monedas:
            if m <= c and M[c - m] + 1 < M[c]:
                M[c] = M[c - m] + 1
    return M

def minimo_monedas(monedas, cantidad):
    minimo = tabla_monedas(monedas, cantidad)[cantidad]
    return None if minimo == inf else minimo

print(minimo_monedas(EUROS, 289), minimo_monedas(EUROS, 388))
print(minimo_monedas([3, 7], 13), minimo_monedas([3, 7], 11))

7 8
3 None


## 2. Reconstruir la solución

Guardar en una segunda tabla la última moneda usada para cada cantidad y, desde la cantidad final, ir restándola para saber qué monedas se dan.

In [2]:
from collections import Counter

def cambio(monedas, cantidad):
    M, ultima = [0] + [inf] * cantidad, [None] * (cantidad + 1)
    for c in range(1, cantidad + 1):
        for m in monedas:
            if m <= c and M[c - m] + 1 < M[c]:
                M[c], ultima[c] = M[c - m] + 1, m
    if M[cantidad] == inf:
        return None
    usadas = []
    while cantidad:
        usadas.append(ultima[cantidad])
        cantidad -= ultima[cantidad]
    return usadas

print(cambio(EUROS, 289))
print(Counter(cambio([1, 3, 4], 6)), Counter(cambio([2, 5], 21)), cambio([3, 7], 11))

[2, 2, 5, 10, 20, 50, 200]
Counter({3: 2}) Counter({2: 3, 5: 3}) None


## 3. Número de formas

Contar de cuántas formas se puede dar una cantidad sin importar el orden: $W[0] = 1$ y, recorriendo las monedas una a una, $W[c] \mathrel{+}= W[c - m]$. Si el bucle de las monedas va por dentro, se cuentan las secuencias ordenadas (1 + 2 y 2 + 1 serían distintas).

In [3]:
def formas(monedas, cantidad):
    W = [1] + [0] * cantidad
    for m in monedas:
        for c in range(m, cantidad + 1):
            W[c] += W[c - m]
    return W[cantidad]

def formas_ordenadas(monedas, cantidad):
    W = [1] + [0] * cantidad
    for c in range(1, cantidad + 1):
        W[c] = sum(W[c - m] for m in monedas if m <= c)
    return W[cantidad]

print(formas([1, 2, 5], 5), formas_ordenadas([1, 2, 5], 5))
print("Formas de dar 2 €:", formas(EUROS, 200), " 5 €:", formas(EUROS, 500))
# con monedas de 1 y 2, las secuencias ordenadas siguen Fibonacci
assert [formas_ordenadas([1, 2], n) for n in range(1, 10)] == [1, 2, 3, 5, 8, 13, 21, 34, 55]

4 9
Formas de dar 2 €: 73682  5 €: 6295434


## 4. Comprobación con fuerza bruta

Comprobar las funciones anteriores con sistemas de monedas aleatorios enumerando todas las combinaciones posibles de cuántas monedas se usan de cada tipo.

In [4]:
import random
from itertools import product

def todas_las_formas(monedas, cantidad):
    return [cuantas for cuantas in product(*(range(cantidad // m + 1) for m in monedas))
            if sum(n * m for n, m in zip(cuantas, monedas)) == cantidad]

random.seed(4)
for _ in range(300):
    sistema = random.sample(range(1, 13), random.randint(1, 4))
    cantidad = random.randint(0, 30)
    combinaciones = todas_las_formas(sistema, cantidad)
    minimo = min((sum(c) for c in combinaciones), default=None)
    assert formas(sistema, cantidad) == len(combinaciones)
    assert minimo_monedas(sistema, cantidad) == minimo
    usadas = cambio(sistema, cantidad)
    if minimo is None:
        assert usadas is None
    else:
        assert len(usadas) == minimo and sum(usadas) == cantidad and set(usadas) <= set(sistema)

## 5. Comparación con el algoritmo voraz

El voraz coge siempre la moneda más grande que quepa. Comparar con la solución óptima y buscar, para varios sistemas, la menor cantidad en la que el voraz falla (da más monedas o no encuentra solución).

In [5]:
def voraz(monedas, cantidad):
    usadas = []
    for m in sorted(monedas, reverse=True):
        while m <= cantidad:
            usadas.append(m)
            cantidad -= m
    return usadas if cantidad == 0 else None

def contraejemplo(monedas, limite=1000):
    M = tabla_monedas(monedas, limite)
    for c in range(1, limite + 1):
        v = voraz(monedas, c)
        if (inf if v is None else len(v)) != M[c]:
            return c
    return None

sistemas = {
    "euro": EUROS,
    "EE. UU.": [1, 5, 10, 25],
    "{1, 3, 4}": [1, 3, 4],
    "{3, 5}": [3, 5],
    "Reino Unido antes de 1971": [1, 3, 6, 12, 24, 30],
}
for nombre, sistema in sistemas.items():
    c = contraejemplo(sistema)
    if c is None:
        print(f"{nombre}: el voraz es óptimo hasta 1000")
    else:
        print(f"{nombre}: falla con {c}: voraz {voraz(sistema, c)}, óptimo {cambio(sistema, c)}")

euro: el voraz es óptimo hasta 1000
EE. UU.: el voraz es óptimo hasta 1000
{1, 3, 4}: falla con 6: voraz [4, 1, 1], óptimo [3, 3]
{3, 5}: falla con 6: voraz None, óptimo [3, 3]
Reino Unido antes de 1971: falla con 48: voraz [30, 12, 6], óptimo [24, 24]
